# Reasoning with Open Source LLMs

This notebook demonstrates how to use an open-source reasoning-capable LLM from Hugging Face. It's created to run on a GPU with 16GB VRAM.

We'll use Mistral-7B-Instruct-v0.3, which is known for its strong reasoning capabilities while being small enough to run on a 16GB GPU.

## Setup and Dependencies

First, let's install the necessary packages:

In [ ]:
# !pip install transformers accelerate bitsandbytes torch einops

## Import Required Libraries

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

## Check GPU Availability

In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Load the Model with Quantization

We'll use 4-bit quantization to fit the model comfortably within 16GB VRAM.

In [ ]:
from huggingface_hub import login


# Make turn to turn this on:
![Image](https://i.imgur.com/9gFDb8a.png)

In [ ]:
login(token="") #TODO add you token here, get one from here https://huggingface.co/settings/tokens, 

In [ ]:
# Configure quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# Model selection
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16
)


In [ ]:
# Print GPU memory usage after loading the model
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## Define Generation Parameters

Here we'll define the parameters for text generation. For reasoning tasks, we want to use a lower temperature and a decent maximum new token length to allow for step-by-step thinking.

In [ ]:
generation_config = {
    "max_new_tokens": 1024,
    "temperature": 0.1,  # Low temperature for more deterministic reasoning
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.1,
    "do_sample": True,   # Enable sampling for some variety in responses
    "pad_token_id": tokenizer.eos_token_id
}

## Create a Helper Function for Inference

Let's create a function that handles prompting the model with appropriate formatting.

In [ ]:
def generate_response(prompt, system_prompt="You are a helpful, honest, and precise assistant."):
    """Generate a response from the model based on the prompt and system prompt."""
    # Format for Mistral-7B-Instruct-v0.2
    formatted_prompt = f"<s>[INST] {system_prompt}\n\n{prompt} [/INST]"
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    # Track token generation time
    start_time = time.time()
    
    # Generate the response
    with torch.no_grad():
        output = model.generate(
            **inputs,
            **generation_config
        )
    
    # Calculate tokens per second
    generation_time = time.time() - start_time
    num_new_tokens = output.shape[1] - inputs['input_ids'].shape[1]
    tokens_per_second = num_new_tokens / generation_time
    
    # Decode the response
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Extract only the model's reply (remove the prompt)
    response = response.split('[/INST]')[-1].strip()
    
    print(f"Generated {num_new_tokens} tokens in {generation_time:.2f} seconds ({tokens_per_second:.2f} tokens/sec)")
    
    return response

## Test with a Simple Reasoning Task

In [ ]:
simple_reasoning_prompt = """
Mark has 5 apples. He gives 2 apples to Sarah. Sarah gives him 3 oranges in return. 
Then Mark buys 4 more apples and 2 more oranges. 
How many apples and oranges does Mark have now? 
Think through this step by step.
"""

response = generate_response(simple_reasoning_prompt)
print(response)

## Complex Reasoning Test: Multi-step Logical Problem

In [ ]:
logical_reasoning_prompt = """
I want you to solve this logical puzzle step by step:

Four friends (Alex, Blake, Casey, and Dana) are deciding where to go on vacation. They are considering four destinations: England, France, Germany, and Italy. From the clues below, determine which person wants to visit which country.

Clues:
1. The person who wants to visit France loves cheese.
2. Dana has been to England before and wants to visit somewhere new.
3. Casey is allergic to cheese.
4. Blake wants to visit Germany or Italy.
5. Alex has never left North America before.
6. The person who wants to visit Italy speaks Italian.
7. Neither Alex nor Blake speaks any foreign languages.
8. Casey speaks French fluently.

For each person, determine which country they want to visit, and explain your reasoning for each deduction.
"""

response = generate_response(logical_reasoning_prompt)
print(response)

## Mathematical Reasoning

In [ ]:
math_reasoning_prompt = """
Please solve this math problem step by step:

A store is having a 15% off sale. Additionally, if you spend more than $100 after the discount, you get an extra $20 off. If I want to buy an item that costs $x, for what values of x will I pay exactly $100 after all discounts are applied?
"""

response = generate_response(math_reasoning_prompt)
print(response)

## Chain-of-Thought Prompting

Let's try using chain-of-thought prompting techniques to enhance reasoning.

In [ ]:
cot_system_prompt = """
You are a helpful assistant that solves problems step-by-step. 
For each problem:
1. Identify what the question is asking for
2. List the relevant facts and constraints
3. Work through the problem systematically
4. Verify your answer
5. State your final answer clearly
"""

cot_prompt = """
The probability of rain on Saturday is 60%. The probability of rain on Sunday is 70%. 
Assuming these events are independent, what is the probability that:
1. It rains on both Saturday and Sunday?
2. It rains on either Saturday or Sunday (or both)?
3. It rains on exactly one of those days?
"""

response = generate_response(cot_prompt, system_prompt=cot_system_prompt)
print(response)

## Load and Try a Different Model (If VRAM allows)

You can also try other models within your 16GB VRAM constraint.

In [ ]:
# Clear the previous model from VRAM
del model
torch.cuda.empty_cache()

In [ ]:
# Load a different model
model_name = "" # Try anything you want
# model_name = "facebook/opt-6.7b" # This gave me very very very bad results

# Load new tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=quantization_config,
    torch_dtype=torch.float16
) 


## Update the helper function to use the response for the new LLM you used

In [ ]:
# def generate_opt_response(prompt, system_prompt="You are a helpful, honest, and precise assistant."):
#     """Generate a response from OPT-6.7B based on the prompt and system prompt."""
#     formatted_prompt = f"System: {system_prompt}\nUser: {prompt}\nAssistant:"
#     
#     inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
#     

#     # Generate the response
#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             **generation_config
#         )
#     
#     # Calculate tokens per second
#     generation_time = time.time() - start_time
#     num_new_tokens = output.shape[1] - inputs['input_ids'].shape[1]
#     tokens_per_second = num_new_tokens / generation_time
#     
#     # Decode the response
#     response = tokenizer.decode(output[0], skip_special_tokens=True)
#     
#     # Extract only the model's reply (remove the prompt)
#     response = response.split("Assistant:")[-1].strip()
#     return response

In [ ]:
def generate_a_response(prompt, system_prompt="You are a helpful, honest, and precise assistant."):
    pass 
    return response

## Test the new one with the Same Reasoning Task

In [ ]:
# Use the same math reasoning prompt
response = generate_a_response(math_reasoning_prompt)
print(response)

## Conclusion

In this notebook, we've demonstrated how to:

1. Load and run reasoning-capable open-source LLMs on a 16GB VRAM GPU
2. Use 4-bit quantization to fit larger models in memory
3. Craft prompts that encourage step-by-step reasoning

You can extend this notebook by:
- Trying other models like Phi-2, TinyLlama, or SOLAR-10.7B-Instruct-v1.0
- Implementing more advanced reasoning techniques like tree-of-thought or self-consistency
- Creating a benchmark of reasoning tasks to evaluate model performance
- Fine-tuning the models on reasoning datasets (requires more VRAM or parameter-efficient techniques like LoRA)